# Introduction

Exploration and illustration of the data in the Shift Project Monde IA from: https://theshiftproject.org/publications/intelligence-artificielle-centres-de-donnees-rapport-final/

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ============================================================
# 1. RAW DATA (macro-level, from your table)
# ============================================================

data = {
    "Region": [
        "World",
        "North America",
        "United States",
        "Central and South America",
        "Brazil",
        "Europe",
        "European Union",
        "Africa",
        "Middle East",
        "Eurasia",
        "Russia",
        "Asia Pacific",
        "China",
        "India",
        "Japan",
        "Southeast Asia",
    ],
    "2010":  [536.4, 112.7, 94.1, 26.7, 12.2, 89.1, 64.5, 25.6, 26.1, 35.8, 29.1, 205.4, 107.0, 27.7, 20.9, 21.3],
    "2023":  [640.7, 112.4, 92.0, 30.1, 14.6, 74.2, 53.2, 34.2, 36.5, 41.4, 33.2, 295.6, 169.6, 46.2, 15.8, 32.5],
    "2024":  [654.2, 113.2, 93.0, 30.4, 14.8, 74.9, 53.5, 34.5, 37.2, 42.3, 33.2, 304.5, 174.5, 46.8, 15.7, 34.0],
    "2035_current": [744.1, 118.6, 98.3, 36.6, 18.1, 73.3, 50.7, 43.2, 48.0, 44.2, 36.4, 358.1, 196.1, 67.0, 13.0, 45.1],
    "2050_current": [838.0, 128.8, 106.3, 46.5, 17.4, 71.0, 46.2, 58.7, 62.8, 44.2, 36.3, 396.3, 191.8, 59.4, 13.2, 59.8],
}

df_raw = pd.DataFrame(data).set_index("Region")

# ============================================================
# 2. BUILD 7 MACRO-REGIONS FROM RAW DATA
# ============================================================

macro = {}

macro["North America"] = df_raw.loc["North America"]
macro["South America"] = df_raw.loc["Central and South America"]
macro["Europe"]        = df_raw.loc["Europe"]
macro["Africa"]        = df_raw.loc["Africa"]
macro["Eurasia"]       = df_raw.loc["Eurasia"]
macro["Asia Pacific"]  = df_raw.loc["Asia Pacific"]
macro["Middle East"]   = df_raw.loc["Middle East"]

df_macro = pd.DataFrame.from_dict(macro, orient="index")
df_macro.index.name = "Region"
df_macro = df_macro.reset_index()

# ============================================================
# 3. LONG FORMAT + INTERPOLATION 2010–2050
# ============================================================

df_long = df_macro.melt(id_vars="Region", var_name="Year_raw", value_name="Value")
df_long["Year"] = df_long["Year_raw"].str.extract(r"(\d+)").astype(int)

years = np.arange(2010, 2051)

interp_rows = []
for region in df_long["Region"].unique():
    sub = df_long[df_long["Region"] == region].set_index("Year").reindex(years)
    sub["Region"] = region
    sub["Value"] = sub["Value"].interpolate()
    sub = sub.reset_index().rename(columns={"index": "Year"})
    interp_rows.append(sub)

df_interp = pd.concat(interp_rows, ignore_index=True)

# ============================================================
# 4. COUNTRY MAPPING (7 MACRO-REGIONS)
# ============================================================

north_america = ["USA", "CAN", "MEX"]

south_america = [
    "ARG","BOL","BRA","CHL","COL","ECU","GUY","PRY","PER","SUR","URY","VEN",
    "BLZ","CRI","SLV","GTM","HND","NIC","PAN","CUB","DOM","HTI","JAM",
    "BHS","BRB","TTO"
]

europe = [
    "AUT","BEL","BGR","HRV","CYP","CZE","DNK","EST","FIN","FRA","DEU","GRC",
    "HUN","IRL","ITA","LVA","LTU","LUX","MLT","NLD","POL","PRT","ROU",
    "SVK","SVN","ESP","SWE",
    "GBR","NOR","CHE","ISL","ALB","MNE","MKD","SRB","BIH","UKR","BLR"
]

africa = [
    "DZA","AGO","BEN","BWA","BFA","BDI","CMR","CPV","CAF","TCD","COM","COG","CIV",
    "COD","DJI","EGY","GNQ","ERI","SWZ","ETH","GAB","GMB","GHA","GIN","GNB","KEN",
    "LSO","LBR","LBY","MDG","MWI","MLI","MRT","MUS","MYT","MAR","MOZ","NAM","NER",
    "NGA","REU","RWA","SHN","STP","SEN","SYC","SLE","SOM","ZAF","SSD","SDN","TZA",
    "TGO","TUN","UGA","ZMB","ZWE"
]

eurasia = ["RUS","KAZ","UZB","TJK","KGZ","TKM","ARM","AZE","GEO"]

asia_pacific = [
    "AUS","NZL","KOR","SGP","MYS","PHL","THA","VNM",
    "CHN","IND","JPN","IDN","BRN","KHM","LAO","MMR","TLS"
]

middle_east = [
    "SAU","ARE","QAT","KWT","OMN","BHR",
    "IRQ","IRN","ISR","JOR","LBN","SYR","YEM","PSE"
]

region_map = {
    "North America": north_america,
    "South America": south_america,
    "Europe":        europe,
    "Africa":        africa,
    "Eurasia":       eurasia,
    "Asia Pacific":  asia_pacific,
    "Middle East":   middle_east,
}

# ============================================================
# 5. COUNTRY-LEVEL TABLE
# ============================================================

country_rows = []
for _, row in df_interp.iterrows():
    reg = row["Region"]
    val = row["Value"]
    year = row["Year"]
    for iso in region_map[reg]:
        country_rows.append({"ISO": iso, "Region": reg, "Year": year, "Value": val})

df_countries = pd.DataFrame(country_rows)

vmin = df_countries["Value"].min()
vmax = df_countries["Value"].max()

# ============================================================
# 6. REGION LABEL CENTROIDS
# ============================================================

region_centroids = {
    "North America": (-100, 50),
    "South America": (-60, -15),
    "Europe":        (15, 55),
    "Africa":        (20, 5),
    "Eurasia":       (70, 45),
    "Asia Pacific":  (110, 10),
    "Middle East":   (45, 28),
}

# ============================================================
# 7. BASE CHOROPLETH
# ============================================================

fig = px.choropleth(
    df_countries,
    locations="ISO",
    color="Value",
    animation_frame="Year",
    color_continuous_scale="OrRd",
    range_color=[vmin, vmax],  # fixed for all years
)

# ============================================================
# 8. ADD MACRO-REGION LABELS (WHITE TEXT) FOR EACH YEAR
# ============================================================

frames = []
unique_years = sorted(df_interp["Year"].unique())

for year in unique_years:
    reg_vals = df_interp[df_interp["Year"] == year]

    scatter = go.Scattergeo(
        lon=[region_centroids[r][0] for r in reg_vals["Region"]],
        lat=[region_centroids[r][1] for r in reg_vals["Region"]],
        text=[f"{v:.1f}" for v in reg_vals["Value"]],
        mode="text",
        textfont=dict(size=16, color="black"),
        hoverinfo="skip",
    )

    frame_index = unique_years.index(year)
    frames.append(
        go.Frame(
            data=[fig.frames[frame_index].data[0], scatter],
            name=str(year),
        )
    )

fig.frames = frames

# first frame labels
reg0 = df_interp[df_interp["Year"] == unique_years[0]]

fig.add_trace(
    go.Scattergeo(
        lon=[region_centroids[r][0] for r in reg0["Region"]],
        lat=[region_centroids[r][1] for r in reg0["Region"]],
        text=[f"{v:.1f}" for v in reg0["Value"]],
        mode="text",
        textfont=dict(size=16, color="black"),
        hoverinfo="skip",
    )
)

fig.update_layout(
    title="Emissions of CO2 by Region (MTons)",
    geo=dict(showframe=False, showcoastlines=True),
    margin=dict(l=0, r=0, t=40, b=0),
)

# If nbformat causes issues in Jupyter, you can use:
# fig.show(renderer="browser")
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1. RAW DATA (macro-level)
# ============================================================

data = {
    "Region": [
        "North America",
        "South America",
        "Europe",
        "Africa",
        "Middle East",
        "Eurasia",
        "Asia Pacific",
    ],
    "2010":  [112.7, 26.7, 89.1, 25.6, 26.1, 35.8, 205.4],
    "2023":  [112.4, 30.1, 74.2, 34.2, 36.5, 41.4, 295.6],
    "2024":  [113.2, 30.4, 74.9, 34.5, 37.2, 42.3, 304.5],
    "2035":  [118.6, 36.6, 73.3, 43.2, 48.0, 44.2, 358.1],
    "2050":  [128.8, 46.5, 71.0, 58.7, 62.8, 44.2, 396.3],
}

df_raw = pd.DataFrame(data)

# ============================================================
# 2. LONG FORMAT + INTERPOLATION 2010–2050
# ============================================================

df_long = df_raw.melt(id_vars="Region", var_name="Year", value_name="Value")
df_long["Year"] = df_long["Year"].astype(int)

years = np.arange(2010, 2051)

rows = []
for region in df_raw["Region"]:
    sub = df_long[df_long["Region"] == region].set_index("Year").reindex(years)
    sub["Region"] = region
    sub["Value"] = sub["Value"].interpolate()
    sub.reset_index(inplace=True)
    rows.append(sub)

df_interp = pd.concat(rows)

# ============================================================
# 3. REGION CENTROIDS
# ============================================================

region_centroids = {
    "North America": (-100, 50),
    "South America": (-60, -15),
    "Europe":        (15, 55),
    "Africa":        (20, 5),
    "Middle East":   (45, 28),
    "Eurasia":       (70, 45),
    "Asia Pacific":  (110, 10),
}

df_interp["lon"] = df_interp["Region"].apply(lambda r: region_centroids[r][0])
df_interp["lat"] = df_interp["Region"].apply(lambda r: region_centroids[r][1])

vmin = df_interp["Value"].min()
vmax = df_interp["Value"].max()

# Fixed circle size for all markers
circle_size = 50

# ============================================================
# 4. BASE MAP (GREY COUNTRIES)
# ============================================================

fig = go.Figure()

fig.update_geos(
    landcolor="#E0E0E0",
    showcountries=True,
    countrycolor="#A0A0A0",
    projection_type="natural earth"
)

# ============================================================
# 5. BUILD ANIMATION FRAMES (CIRCLES + LABELS + YEAR)
# ============================================================

frames = []
years_sorted = sorted(df_interp["Year"].unique())

for year in years_sorted:
    sub = df_interp[df_interp["Year"] == year]

    circles = go.Scattergeo(
        lon=sub["lon"],
        lat=sub["lat"],
        mode="markers",
        marker=dict(
            size=circle_size,
            color=sub["Value"],
            colorscale="RdBu",
            reversescale=True,  # low=blue, high=red
            cmin=vmin,
            cmax=vmax,
            opacity=0.85,
            line=dict(width=0),
        ),
        hoverinfo="skip"
    )

    labels = go.Scattergeo(
        lon=sub["lon"],
        lat=sub["lat"],
        mode="text",
        text=[f"{v:.1f}" for v in sub["Value"]],
        textfont=dict(size=14, color="white", family="Arial Black"),
        hoverinfo="skip"
    )

    # YEAR ABOVE THE MAP FRAME
    year_annotation = go.layout.Annotation(
        x=0.5, y=1.25,            # ABOVE THE MAP
        xref="paper", yref="paper",
        text=f"Emissions of CO2 by Region (MTons) - {year}",
        showarrow=False,
        font=dict(size=40, color="black", family="Arial Black")
    )

    frames.append(
        go.Frame(
            data=[circles, labels],
            layout=go.Layout(annotations=[year_annotation]),
            name=str(year)
        )
    )

# ============================================================
# 6. INITIAL FRAME
# ============================================================

year0 = years_sorted[0]
sub0 = df_interp[df_interp["Year"] == year0]

circles0 = go.Scattergeo(
    lon=sub0["lon"],
    lat=sub0["lat"],
    mode="markers",
    marker=dict(
        size=circle_size,
        color=sub0["Value"],
        colorscale="RdBu",
        reversescale=True,
        cmin=vmin,
        cmax=vmax,
        opacity=0.85,
        line=dict(width=0),
    ),
    hoverinfo="skip"
)

labels0 = go.Scattergeo(
    lon=sub0["lon"],
    lat=sub0["lat"],
    mode="text",
    text=[f"{v:.1f}" for v in sub0["Value"]],
    textfont=dict(size=14, color="white", family="Arial Black"),
    hoverinfo="skip"
)

# Initial annotation above map
initial_annotation = go.layout.Annotation(
    x=0.5, y=1.25,
    xref="paper", yref="paper",
    text=f"Emissions of CO2 by Region (MTons) - {year0}",
    showarrow=False,
    font=dict(size=40, color="black", family="Arial Black")
)

fig.add_trace(circles0)
fig.add_trace(labels0)

fig.update_layout(annotations=[initial_annotation])

fig.frames = frames

# ============================================================
# 7. LAYOUT WITH PLAY BUTTON
# ============================================================

fig.update_layout(
    title="Emissions of CO2 by Region (MTons)",
    showlegend=False,
    geo=dict(showland=True),
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [
                        None,
                        {"frame": {"duration": 60}, "fromcurrent": True}
                    ]
                }
            ]
        }
    ]
)

fig.show()


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
data_path = "../data/vody.xlsx"
df = pd.read_excel(data_path)

In [5]:
df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,Variable,Description,2025.00,2030.00,Commentaires,Liens utiles,Incertitude,Valeur moyenne 2024,Valeur optimiste 2024,Valeur conservatrice 2024,Commentaires
1,NaN,n_acc(τ),Nombre d'accélérateurs d’IA sur le marché cons...,9068000.00,61000000.00,Voir onglet [Input - n_acc] : La valeur moyenn...,"[4], [5], [6]",Forte,4220000,3500000,5000000,Estimation à mi-2024 à partir de données epoch...
2,NaN,TDP,Thermal Design Power : Transfert thermique ver...,0.70,1.10,En prenant l'hypothèse d'une durée de vie d'un...,"[7], [8]",Faible,0.7,0.3,0.7,Valeur moyenne : TDP NVidia H100\nValeur optim...
3,NaN,Overhead_serveuracc,Puissance énergétique sollicitée par le serveu...,1.82,1.82,Un serveur NVidia DGX H100 de 8 GPU H100 a un ...,"[15], [16]",Faible,2,1.82,2.7,Valeur moyenne : Ratio TDP DGX H100 / TDP 8 H1...
4,NaN,H,Nombre d’heures dans une année,8766.00,8766.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
